In [ ]:
import pandas as pd

df = pd.read_csv('phishing_email.csv')  # load filename
print(df.head())
print(df.columns)

                                       text_combined  label
0  hpl nom may 25 2001 see attached file hplno 52...      0
1  nom actual vols 24 th forwarded sabrae zajac h...      0
2  enron actuals march 30 april 1 201 estimated a...      0
3  hpl nom may 30 2001 see attached file hplno 53...      0
4  hpl nom june 1 2001 see attached file hplno 60...      0
Index(['text_combined', 'label'], dtype='object')


In [ ]:
print(df['label'].value_counts())

label
1    42891
0    39595
Name: count, dtype: int64


In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_words = 20000    # top 20k words
max_len = 300        # max sequence length

tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(df['text_combined'])

X = tokenizer.texts_to_sequences(df['text_combined'])
X = pad_sequences(X, maxlen=max_len)

y = df['label'].values

In [ ]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

model = Sequential([
    Embedding(input_dim=max_words, output_dim=128, input_length=max_len),
    LSTM(64),
    Dropout(0.5),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')  # binary output
])

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

model.fit(X_train, y_train,
          validation_split=0.1,
          epochs=5,
          batch_size=128,
          verbose=2)

Epoch 1/5


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


464/464 - 294s - 633ms/step - accuracy: 0.9608 - loss: 0.1138 - val_accuracy: 0.9836 - val_loss: 0.0462
Epoch 2/5
464/464 - 316s - 681ms/step - accuracy: 0.9931 - loss: 0.0235 - val_accuracy: 0.9861 - val_loss: 0.0425
Epoch 3/5
464/464 - 315s - 678ms/step - accuracy: 0.9966 - loss: 0.0115 - val_accuracy: 0.9868 - val_loss: 0.0446
Epoch 4/5
464/464 - 321s - 691ms/step - accuracy: 0.9982 - loss: 0.0064 - val_accuracy: 0.9856 - val_loss: 0.0572
Epoch 5/5
464/464 - 318s - 685ms/step - accuracy: 0.9989 - loss: 0.0046 - val_accuracy: 0.9882 - val_loss: 0.0559


In [ ]:
loss, accuracy = model.evaluate(X_test, y_test)
print(f'Test Accuracy: {accuracy:.4f}')

516/516 ━━━━━━━━━━━━━━━━━━━━ 26s 50ms/step - accuracy: 0.9883 - loss: 0.0521
Test Accuracy: 0.9882
